In [0]:
dbutils.widgets.text("catalog","sj20030614")
catalog = dbutils.widgets.get("catalog")

#bronze.stations 

In [0]:
#sql
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.bronze.stations AS
SELECT *,
    _metadata.file_path AS _source_file,
    current_timestamp() AS _ingested_at
 FROM read_files('/Volumes/{catalog}/landing/raw/stations/stations.csv')
""")

In [0]:
# COUNT ROWS IN BRONZE.STATION
df = spark.sql(f"""
    SELECT COUNT(*) FROM {catalog}.bronze.stations
""")
display(df)


#bronze.tariffs

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.bronze.tariffs AS
SELECT *,
    _metadata.file_path AS _source_file,
    current_timestamp() AS _ingested_at
 FROM read_files('/Volumes/{catalog}/landing/raw/tariffs/tariffs.csv')
""")

In [0]:
df = spark.sql(f"""
    SELECT COUNT(*) FROM {catalog}.bronze.tariffs
""")
display(df)

#bronze.members

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.bronze.members COMMENT 'IT IS A MEMMBERS TABLE'
""")

In [0]:
spark.sql(f"""
COPY INTO {catalog}.bronze.members
FROM (
    SELECT *,
    _metadata.file_path AS _source_file,
    current_timestamp() AS _ingested_at
    FROM '/Volumes/{catalog}/landing/raw/members/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')          
""")

In [0]:
df = spark.sql(f"""
    SELECT COUNT(*) FROM {catalog}.bronze.members
""")
display(df)

#bronze.chargers

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.bronze.chargers COMMENT "IT IS A CHARGERS TABLE"
""")

In [0]:
spark.sql(f"""
COPY INTO {catalog}.bronze.chargers
FROM (
    SELECT *,
    _metadata.file_path AS _source_file,
    current_timestamp() AS _ingested_at
    FROM '/Volumes/{catalog}/landing/raw/chargers/'
)
FILEFORMAT = JSON
FORMAT_OPTIONS ('header' = 'true','multiline' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')          
""")

In [0]:
df = spark.sql(f"""
    SELECT COUNT(*) FROM {catalog}.bronze.chargers
""")
display(df)

In [0]:
df = spark.table('sj20030614.bronze.chargers')
df.printSchema()

#bronze.sessions

In [0]:
from pyspark.sql.functions import *

df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format","json") \
    .option("cloudFiles.schemaLocation",f"/Volumes/{catalog}/ops/pipeline_state/s_schema/") \
    .option("cloudFiles.schemaEvolutionMode","addNewColumns") \
    .option("multiline","true") \
    .option("cloudFiles.inferColumnTypes", "true") \
    .load(f"/Volumes/{catalog}/landing/raw/sessions/")

df = df.withColumn("_source_file",col("_metadata.file_path")) \
    .withColumn("_ingested_at", current_timestamp())

df.writeStream \
    .format("delta") \
    .option("checkpointLocation", f"/Volumes/{catalog}/ops/pipeline_state/s_checkpoint") \
    .trigger(availableNow=True) \
    .option('mergeSchema','true') \
    .toTable(f"{catalog}.bronze.sessions")